In [ ]:
import os
from openai import AzureOpenAI

endpoint = "https://conversationalai12.openai.azure.com/"
model_name = "gpt-4o-mini"
deployment = "gpt-4o-mini"

subscription_key = "9Cyy3yKHaukqFUy1DPNAPkxY3HnQkeCSA4EraiLqYPv7JEoArn2RJQQJ99BDACYeBjFXJ3w3AAABACOGmXd4"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

question= "Please propose a hard, challenging question to assess someone's IQ. Respond only with the question."


response = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": question,
        }
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model=deployment
)

question1= (response.choices[0].message.content)
print(question1)
print("********************************")

response1 = client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": question1,
        }
    ],
    max_tokens=4096,
    temperature=1.0,
    top_p=1.0,
    model=deployment
)
from IPython.display import Markdown, display
display((Markdown(response1.choices[0].message.content)))


If a train leaves City A heading towards City B at a speed of 60 miles per hour, and at the same time, another train leaves City B heading towards City A at a speed of 90 miles per hour, and the distance between City A and City B is 300 miles, how long will it take for the two trains to meet?
********************************


To find out how long it takes for the two trains to meet, we can add their speeds together because they are moving towards each other.

The speed of Train A (from City A to City B) is 60 miles per hour, and the speed of Train B (from City B to City A) is 90 miles per hour. 

Total speed of both trains:
\[
60 \text{ miles per hour} + 90 \text{ miles per hour} = 150 \text{ miles per hour}
\]

Next, the distance between the two cities is 300 miles. We can find the time it takes for the two trains to meet by using the formula:
\[
\text{Time} = \frac{\text{Distance}}{\text{Speed}}
\]

Substituting the values we have:
\[
\text{Time} = \frac{300 \text{ miles}}{150 \text{ miles per hour}} = 2 \text{ hours}
\]

Therefore, it will take **2 hours** for the two trains to meet.

In [14]:
# --- Minimal agentic loop with Azure OpenAI (Planner + Tool + Executor) ---
# Prereqs:
#   pip install openai
#   Set env vars: AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, AZURE_OPENAI_API_VERSION, AZURE_OPENAI_DEPLOYMENT

import os, math, json
from openai import AzureOpenAI

endpoint = "https://conversationalai12.openai.azure.com/"
model_name = "gpt-4o-mini"
deployment = "gpt-4o-mini"

subscription_key = "9Cyy3yKHaukqFUy1DPNAPkxY3HnQkeCSA4EraiLqYPv7JEoArn2RJQQJ99BDACYeBjFXJ3w3AAABACOGmXd4"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# 1) Define a simple TOOL the model can call (calculator as example)
def calc(expr: str) -> str:
    # VERY basic; production should use a safe evaluator
    try:
        return str(eval(expr, {"__builtins__": {}}, {"sqrt": math.sqrt, "pow": pow}))
    except Exception as e:
        return f"ERROR: {e}"

tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a simple math expression. Supports + - * / pow() sqrt().",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    }
]

# 2) Messages = memory/state
goal = "Propose one tough IQ-style question, then verify its answer with the calculator if needed, and output the final Q and A."
messages = [
    {"role": "system",
     "content": "You are a Planner-Executor agent. Plan briefly, use tools when needed, and stop when the user gets a final answer."},
    {"role": "user", "content": goal},
]

# 3) Loop with tool use + termination rules
MAX_STEPS = 5
for step in range(1, MAX_STEPS + 1):
    resp = client.chat.completions.create(
        model=deployment,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        temperature=0.3,
        max_tokens=300,
    )

    msg = resp.choices[0].message

    # If the model wants to call a tool
    if msg.tool_calls:
        for call in msg.tool_calls:
            if call.function.name == "calculator":
                args = json.loads(call.function.arguments)
                result = calc(args.get("expression", ""))
                messages.append(
                    {"role": "tool", "tool_call_id": call.id, "name": "calculator", "content": result}
                )
        continue  # let the model observe tool results next step

    # No tool call -> candidate final answer
    content = msg.content or ""
    messages.append({"role": "assistant", "content": content})

    # Simple stop condition: model claims final answer or we’ve done enough steps
    if "Final answer" in content or step == MAX_STEPS:
        print(content)
        break


Here's a tough IQ-style question:

**Question:** A farmer has 17 sheep, and all but 9 die. How many sheep does the farmer have left?

Now, let's verify the answer.

**Answer:** The farmer has 9 sheep left, as the question states "all but 9 die," meaning 9 sheep are still alive.

Final Q and A:

**Q:** A farmer has 17 sheep, and all but 9 die. How many sheep does the farmer have left?  
**A:** 9 sheep.


In [36]:
# --- Basic agentic loop: plan → (optional) act via tool → observe → repeat → FINAL ---
# Prereqs:
#   pip install -U openai
#   Set env vars: AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, AZURE_OPENAI_API_VERSION, AZURE_OPENAI_DEPLOYMENT

import os, json, math
from openai import AzureOpenAI

endpoint = "https://conversationalai12.openai.azure.com/"
model_name = "gpt-4o-mini"
deployment = "gpt-4o-mini"

subscription_key = "9Cyy3yKHaukqFUy1DPNAPkxY3HnQkeCSA4EraiLqYPv7JEoArn2RJQQJ99BDACYeBjFXJ3w3AAABACOGmXd4"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# 1) Tooling: one simple function (swap in your D365/SAP/Coupa later)
def calculator(expression: str) -> str:
    try:
        # Safe-ish eval surface (allow only math ops/funcs you bless)
        allowed = {"sqrt": math.sqrt, "pow": pow}
        return str(eval(expression, {"__builtins__": {}}, allowed))
    except Exception as e:
        return f"ERROR: {e}"

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a math expression. Supports + - * / pow() sqrt().",
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    }
]

# 2) System contract: how to stop
SYSTEM = (
    "You are a Planner-Executor agent. If you need a tool, call it via function calling. "
    "When you have the final answer, reply with a single message starting with 'FINAL:' "
    "and DO NOT call any tools in that turn."
)

goal = "Propose one tough IQ-style question, verify the correct answer with the calculator if needed, and output just the final Q&A."

# 3) State (memory)
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": goal},
]

# 4) LOOP: this is where it starts, runs, and stops
MAX_STEPS = 5
for step in range(1, MAX_STEPS + 1):
    print(f"[step {step}] → calling model")
    #print(f"[step {step}] tool_calls? {bool(msg.tool_calls)}")
    #print("messages tail:", messages[-3:])  # last few messages


    resp = client.chat.completions.create(
        model=deployment,
        messages=messages,
        tools=TOOLS,
        tool_choice="auto",
        temperature=0.2,
        max_tokens=400,
    )
    msg = resp.choices[0].message

    print( msg )
    # Branch A: the model wants a tool → you execute → append result → NEXT ITERATION
    if msg.tool_calls:
        messages.append({"role": "assistant", "tool_calls": msg.tool_calls})  # keep trace
        for call in msg.tool_calls:
            if call.function.name == "calculator":
                args = json.loads(call.function.arguments or "{}")
                print(args)
                result = calculator(args.get("expression", ""))
                # return tool output to the model (observation)
                print('&&&&&&')
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "name": "calculator",
                    "content": result
                })
        continue  # go to next step so the model can observe tool outputs

    # Branch B: no tool call → could be thinking or final
    content = (msg.content or "").strip()
    messages.append({"role": "assistant", "content": content})

    if content.startswith("FINAL:"):
        # TERMINATION: clean exit
        print("\n=== FINAL OUTPUT ===")
        print(content[len("FINAL:"):].strip())
        break

    # If not FINAL and no tool call, let it think one more step; budget protects us.
else:
    # HARD STOP: we exhausted the step budget
    print("\n[guardrail] Max steps reached without FINAL. Review the prompt or raise MAX_STEPS.")


[step 1] → calling model
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_xwlc59CznIa0W783l6cKkoon', function=Function(arguments='{"expression":"(8 * 7) - (3 * 5) + (12 / 4)"}', name='calculator'), type='function')])
{'expression': '(8 * 7) - (3 * 5) + (12 / 4)'}
&&&&&&
[step 2] → calling model
ChatCompletionMessage(content='FINAL: \n\n**Question:** If you multiply 8 by 7, subtract the product of 3 and 5, and then add the result of 12 divided by 4, what is the final result?\n\n**Answer:** 44', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

=== FINAL OUTPUT ===
**Question:** If you multiply 8 by 7, subtract the product of 3 and 5, and then add the result of 12 divided by 4, what is the final result?

**Answer:** 44


register_tools()
SYSTEM ← policy_text
messages ← [system(SYSTEM), user(request)]
budgets ← {steps, tokens, ms}; start_timer(); total_tokens=0

for step in 1..budgets.steps:
    resp ← LLM(messages, tools, tool_choice="auto", max_tokens=low)
    total_tokens += resp.usage.tokens; if over budget → stop
    if time_over(budgets.ms) → stop

    if resp.message.tool_calls:
        append(messages, assistant(tool_calls))
        for call in tool_calls:
            args ← json_parse(call.arguments)
            result ← dispatch_tool(call.name, args)   # returns JSON + call_id
            append(messages, tool(call.id, call.name, result))
        continue

    content ← resp.message.content
    append(messages, assistant(content))
    if startswith(content, "FINAL_JSON:"):
        return content

    prune(messages)  # keep context small

return "guardrail hit"


Agentic AI Realtime

In [39]:
# spare_parts_agent.py
# Agentic spare-parts planner: plan → (optional) call tools → observe → FINAL_JSON
# Requirements: pip install -U openai
# Env: AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, AZURE_OPENAI_API_VERSION, AZURE_OPENAI_DEPLOYMENT

from __future__ import annotations
import os, json, time, math, uuid
from dataclasses import dataclass
from typing import Any, Dict, Tuple, Callable, Optional, List
from openai import AzureOpenAI

# ============== Azure OpenAI client ==============
endpoint = "https://conversationalai12.openai.azure.com/"
model_name = "gpt-4o-mini"
DEPLOYMENT = "gpt-4o-mini"

subscription_key = "9Cyy3yKHaukqFUy1DPNAPkxY3HnQkeCSA4EraiLqYPv7JEoArn2RJQQJ99BDACYeBjFXJ3w3AAABACOGmXd4"
api_version = "2024-12-01-preview"

CLIENT = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# ============== Mock backend (swap with real APIs later) ==============
MOCK_WORK_ORDERS = {
    "WO-981": {"asset_id": "PUMP-22", "site": "K", "need_by": "2025-08-30", "parts": [{"part_id": "PS-200", "qty": 2}]},
}
MOCK_BOM = {
    "PUMP-22": [{"part_id": "PS-200", "qty": 2}, {"part_id": "GL-010", "qty": 1}],
}
MOCK_STOCK = {
    # site -> part -> record
    "K": {"PS-200": {"on_hand": 0, "reserved": 0, "bins": ["A1"]}, "PS-200A": {"on_hand": 1, "reserved": 0, "bins": ["B4"]}},
    "M": {"PS-200": {"on_hand": 3, "reserved": 0, "bins": ["Z9"]}, "PS-200A": {"on_hand": 10, "reserved": 1, "bins": ["M2"]}},
}
MOCK_LEAD = {"PS-200": {"SupplierX": 10}, "PS-200A": {"SupplierX": 4}}
MOCK_PRICE = {"PS-200": [(1, 1200.0), (5, 1100.0)], "PS-200A": [(1, 1000.0), (5, 900.0)]}
MOCK_SUBS = {"PS-200": [{"part_id": "PS-200A", "notes": "Qualified alt"}]}
MOCK_FAILURE = {("PUMP-22", "PS-200"): {"mtbf_days": 240, "events_12m": 2}}

# ============== Tool registry & dispatcher ==============
@dataclass
class ToolCallResult:
    call_id: str
    content: str  # JSON string returned to the model

class ToolRegistry:
    def __init__(self):
        self._registry: Dict[str, Callable[[Dict[str, Any]], ToolCallResult]] = {}
    def register(self, name: str, fn: Callable[[Dict[str, Any]], ToolCallResult]):
        self._registry[name] = fn
    def dispatch(self, name: str, args: Dict[str, Any]) -> ToolCallResult:
        if name not in self._registry:
            return ToolCallResult(call_id=str(uuid.uuid4()), content=json.dumps({"ok": False, "error": f"Unknown tool {name}"}))
        return self._registry[name](args)

TOOLS_SPECS = [
    {
        "type": "function",
        "function": {
            "name": "get_work_order",
            "description": "Fetch work order: asset, site, parts, need-by.",
            "parameters": {"type": "object", "properties": {"wo_id": {"type": "string"}}, "required": ["wo_id"]},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_bom",
            "description": "Get BOM for an asset.",
            "parameters": {"type": "object", "properties": {"asset_id": {"type": "string"}}, "required": ["asset_id"]},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_stock",
            "description": "On-hand, reserved, locations for a part at a site.",
            "parameters": {
                "type": "object",
                "properties": {"part_id": {"type": "string"}, "site": {"type": "string"}},
                "required": ["part_id", "site"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_lead_time",
            "description": "Lead time (days) for a part+supplier.",
            "parameters": {
                "type": "object",
                "properties": {"part_id": {"type": "string"}, "supplier": {"type": "string"}},
                "required": ["part_id", "supplier"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "suggest_substitute",
            "description": "Qualified alternates and compatibility notes.",
            "parameters": {"type": "object", "properties": {"part_id": {"type": "string"}}, "required": ["part_id"]},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "transfer_options",
            "description": "Find donor sites that can transfer the part.",
            "parameters": {
                "type": "object",
                "properties": {
                    "part_id": {"type": "string"},
                    "needed_qty": {"type": "integer"},
                    "to_site": {"type": "string"},
                    "from_sites": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["part_id", "needed_qty", "to_site", "from_sites"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "price_breaks",
            "description": "Supplier price tiers for a part.",
            "parameters": {"type": "object", "properties": {"part_id": {"type": "string"}}, "required": ["part_id"]},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "failure_history",
            "description": "MTBF and recurrence risk for asset+part.",
            "parameters": {
                "type": "object",
                "properties": {"asset_id": {"type": "string"}, "part_id": {"type": "string"}},
                "required": ["asset_id", "part_id"],
            },
        },
    },
]

REG = ToolRegistry()

def _wrap_result(payload: Dict[str, Any]) -> ToolCallResult:
    cid = str(uuid.uuid4())
    payload = dict(payload)
    payload["call_id"] = cid
    return ToolCallResult(call_id=cid, content=json.dumps(payload))

def tool_get_work_order(args: Dict[str, Any]) -> ToolCallResult:
    wo = MOCK_WORK_ORDERS.get(args.get("wo_id"))
    if not wo:
        return _wrap_result({"ok": False, "error": "WO not found"})
    return _wrap_result({"ok": True, "wo": wo, "wo_id": args["wo_id"]})

def tool_get_bom(args: Dict[str, Any]) -> ToolCallResult:
    bom = MOCK_BOM.get(args.get("asset_id"), [])
    return _wrap_result({"ok": True, "asset_id": args.get("asset_id"), "bom": bom})

def tool_check_stock(args: Dict[str, Any]) -> ToolCallResult:
    site, part = args.get("site"), args.get("part_id")
    rec = MOCK_STOCK.get(site, {}).get(part)
    if not rec:
        return _wrap_result({"ok": True, "site": site, "part_id": part, "on_hand": 0, "reserved": 0, "bins": []})
    return _wrap_result({"ok": True, "site": site, "part_id": part, **rec})

def tool_check_lead_time(args: Dict[str, Any]) -> ToolCallResult:
    part, supplier = args.get("part_id"), args.get("supplier")
    days = MOCK_LEAD.get(part, {}).get(supplier)
    if days is None:
        return _wrap_result({"ok": False, "error": "No lead time data", "part_id": part, "supplier": supplier})
    return _wrap_result({"ok": True, "part_id": part, "supplier": supplier, "lead_time_days": days})

def tool_suggest_substitute(args: Dict[str, Any]) -> ToolCallResult:
    subs = MOCK_SUBS.get(args.get("part_id"), [])
    return _wrap_result({"ok": True, "part_id": args.get("part_id"), "subs": subs})

def tool_transfer_options(args: Dict[str, Any]) -> ToolCallResult:
    part, needed, to_site, from_sites = args.get("part_id"), int(args.get("needed_qty", 0)), args.get("to_site"), args.get("from_sites") or []
    donors = []
    for s in from_sites:
        stock = MOCK_STOCK.get(s, {}).get(part, {"on_hand": 0, "reserved": 0})
        free = max(0, stock.get("on_hand", 0) - stock.get("reserved", 0))
        if free > 0:
            donors.append({"from_site": s, "available": free, "eta_hours": 18 if s != to_site else 0})
    donors.sort(key=lambda d: d["eta_hours"])
    return _wrap_result({"ok": True, "part_id": part, "needed_qty": needed, "to_site": to_site, "options": donors})

def tool_price_breaks(args: Dict[str, Any]) -> ToolCallResult:
    tiers = MOCK_PRICE.get(args.get("part_id"), [])
    return _wrap_result({"ok": True, "part_id": args.get("part_id"), "tiers": tiers})

def tool_failure_history(args: Dict[str, Any]) -> ToolCallResult:
    data = MOCK_FAILURE.get((args.get("asset_id"), args.get("part_id")), {"mtbf_days": None, "events_12m": 0})
    return _wrap_result({"ok": True, "asset_id": args.get("asset_id"), "part_id": args.get("part_id"), **data})

# Register tools
REG.register("get_work_order", tool_get_work_order)
REG.register("get_bom", tool_get_bom)
REG.register("check_stock", tool_check_stock)
REG.register("check_lead_time", tool_check_lead_time)
REG.register("suggest_substitute", tool_suggest_substitute)
REG.register("transfer_options", tool_transfer_options)
REG.register("price_breaks", tool_price_breaks)
REG.register("failure_history", tool_failure_history)

# ============== System policy ==============
SYSTEM = (
    "You are a Spare-Parts Planning Agent for maintenance operations.\n"
    "Rules:\n"
    "1) Never invent numbers or facts—use tools for every numeric claim.\n"
    "2) If you mention stock, lead time, price, or substitutions, you MUST call the relevant tool and include citations (tool call IDs).\n"
    "3) Do NOT perform write actions. Only propose an ACTION PLAN; humans approve execution.\n"
    "4) When finished, output ONLY:\n"
    "FINAL_JSON: {\"summary\":\"...\",\"actions\":[{\"tool\":\"...\",\"args\":{...}}],\"citations\":[{\"source\":\"tool\",\"id\":\"...\"}],\"risks\":[\"...\"],\"next_step\":\"approve|reject|refine\"}\n"
    "Do not emit FINAL_JSON until all needed tools have been called and observed."
)

# ============== Agent controller ==============
@dataclass
class Budgets:
    max_steps: int = 6
    max_tokens_total: int = 3000
    max_ms: int = 6000

def run_agent(user_request: str, budgets: Budgets = Budgets()) -> Optional[str]:
    """Return FINAL_JSON string or None if guardrail hit."""
    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_request},
    ]
    total_tokens = 0
    t0 = time.perf_counter()

    for step in range(1, budgets.max_steps + 1):
        # Time guard
        if (time.perf_counter() - t0) * 1000 > budgets.max_ms:
            print("[guardrail] time budget exceeded")
            return None

        # Model call
        resp = CLIENT.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            tools=TOOLS_SPECS,
            tool_choice="auto",
            temperature=0.2,
            max_tokens=500,
        )
        usage = getattr(resp, "usage", None)
        if usage:
            total_tokens += (usage.prompt_tokens or 0) + (usage.completion_tokens or 0)
            if total_tokens > budgets.max_tokens_total:
                print("[guardrail] token budget exceeded")
                return None

        msg = resp.choices[0].message

        # Tool branch
        if msg.tool_calls:
            messages.append({"role": "assistant", "tool_calls": msg.tool_calls})
            for call in msg.tool_calls:
                name = call.function.name
                args = json.loads(call.function.arguments or "{}")
                # Dispatch mock tool
                result = REG.dispatch(name, args)
                # Return JSON content (includes call_id) back to the model
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "name": name,
                    "content": result.content
                })
            continue

        # Text branch
        content = (msg.content or "").strip()
        messages.append({"role": "assistant", "content": content})

        if content.startswith("FINAL_JSON:"):
            # Done
            print(content)  # Full artifact for logs
            return content

        # Else: allow another reasoning turn

    print("[guardrail] max steps reached without FINAL_JSON")
    return None

# ============== Example run ==============
if __name__ == "__main__":
    # Example: a planner asks about a WO need
    query = "WO-981 needs 2 x PS-200 at Site K by Friday. Propose fastest vs lowest-cost plan."
    out = run_agent(query)
    if not out:
        print("No final output. Refine the request or raise budgets.")


[guardrail] token budget exceeded
No final output. Refine the request or raise budgets.


In [40]:
from __future__ import annotations
import os, json, time, uuid, re
from dataclasses import dataclass
from typing import Any, Dict, Callable, List, Optional
from openai import AzureOpenAI

# ---------- Azure client ----------

SCENARIO = os.getenv("SCENARIO", "TRANSFER").upper()

endpoint = "https://conversationalai12.openai.azure.com/"
model_name = "gpt-4o-mini"
DEPLOYMENT = "gpt-4o-mini"

subscription_key = "9Cyy3yKHaukqFUy1DPNAPkxY3HnQkeCSA4EraiLqYPv7JEoArn2RJQQJ99BDACYeBjFXJ3w3AAABACOGmXd4"
api_version = "2024-12-01-preview"

CLIENT = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# ---------- Fixtures (LLM stays live; data comes from tools) ----------
def load_fixture(s: str) -> Dict[str, Any]:
    data = {
        "WORK_ORDERS": {
            "WO-981": {"asset_id":"PUMP-22","site":"K","need_by":"2025-08-30","parts":[{"part_id":"PS-200","qty":2}]}
        },
        "BOM": {"PUMP-22":[{"part_id":"PS-200","qty":2}]},
        "SUBS": {"PS-200":[{"part_id":"PS-200A","notes":"Qualified alt"}]},
        "LEAD": {"PS-200":{"SupplierX":10},"PS-200A":{"SupplierX":4}},
        "PRICE":{"PS-200":[(1,1200.0)],"PS-200A":[(1,1000.0)]},
        "STOCK":{"K":{},"M":{}}
    }
    if s=="ONHAND_OK":
        data["STOCK"]["K"]["PS-200"] = {"on_hand":5,"reserved":0,"bins":["A1"]}
    elif s=="NEED_SUB":
        data["STOCK"]["K"]["PS-200"]  = {"on_hand":0,"reserved":0,"bins":["A1"]}
        data["STOCK"]["K"]["PS-200A"] = {"on_hand":3,"reserved":0,"bins":["B4"]}
    elif s=="TRANSFER":
        data["STOCK"]["K"]["PS-200"]  = {"on_hand":0,"reserved":0,"bins":["A1"]}
        data["STOCK"]["K"]["PS-200A"] = {"on_hand":1,"reserved":0,"bins":["B4"]}
        data["STOCK"]["M"]["PS-200A"] = {"on_hand":10,"reserved":1,"bins":["M2"]}
    elif s=="BACKORDER":
        data["STOCK"]["K"]["PS-200"] = {"on_hand":0,"reserved":0,"bins":["A1"]}
        data["STOCK"]["M"]["PS-200"] = {"on_hand":0,"reserved":0,"bins":[]}
        data["SUBS"]["PS-200"] = []
        data["LEAD"]["PS-200"]["SupplierX"] = 14
    else:
        raise ValueError("Unknown SCENARIO")
    return data

DATA = load_fixture(SCENARIO)

# ---------- Tool registry ----------
@dataclass
class ToolCallResult:
    call_id: str
    content: str

def _wrap(payload: Dict[str, Any]) -> ToolCallResult:
    cid = str(uuid.uuid4())
    obj = dict(payload); obj["call_id"] = cid
    return ToolCallResult(call_id=cid, content=json.dumps(obj, separators=(",",":")))

class ToolRegistry:
    def __init__(self): self._reg: Dict[str, Callable[[Dict[str, Any]], ToolCallResult]] = {}
    def register(self, name: str, fn: Callable[[Dict[str, Any]], ToolCallResult]): self._reg[name]=fn
    def dispatch(self, name: str, args: Dict[str, Any]) -> ToolCallResult:
        fn = self._reg.get(name)
        return _wrap({"ok":False,"error":f"Unknown tool {name}"}) if not fn else fn(args)

REG = ToolRegistry()

# ---------- Tools (fixture-backed, read-only) ----------
def t_get_work_order(args: Dict[str, Any]) -> ToolCallResult:
    wo_id = args.get("wo_id"); wo = DATA["WORK_ORDERS"].get(wo_id)
    return _wrap({"ok":False,"error":"WO not found","wo_id":wo_id}) if not wo else _wrap({"ok":True,"wo_id":wo_id,"wo":wo})

def t_get_bom(args: Dict[str, Any]) -> ToolCallResult:
    asset = args.get("asset_id"); return _wrap({"ok":True,"asset_id":asset,"bom":DATA["BOM"].get(asset,[])})

def t_check_stock(args: Dict[str, Any]) -> ToolCallResult:
    site, part = args.get("site"), args.get("part_id")
    rec = DATA["STOCK"].get(site,{}).get(part)
    return _wrap({"ok":True,"site":site,"part_id":part,"on_hand":0,"reserved":0,"bins":[]}) if not rec else _wrap({"ok":True,"site":site,"part_id":part,**rec})

def t_suggest_substitute(args: Dict[str, Any]) -> ToolCallResult:
    part = args.get("part_id"); return _wrap({"ok":True,"part_id":part,"subs":DATA["SUBS"].get(part,[])})

def t_transfer_options(args: Dict[str, Any]) -> ToolCallResult:
    part, need, to_site = args.get("part_id"), int(args.get("needed_qty",0)), args.get("to_site")
    donors=[]
    for site, parts in DATA["STOCK"].items():
        if site==to_site: continue
        rec = parts.get(part);
        if not rec: continue
        free = max(0, rec.get("on_hand",0)-rec.get("reserved",0))
        if free>0: donors.append({"from_site":site,"available":free,"eta_hours":18})
    donors.sort(key=lambda d:d["eta_hours"])
    return _wrap({"ok":True,"part_id":part,"needed_qty":need,"to_site":to_site,"options":donors})

def t_check_lead_time(args: Dict[str, Any]) -> ToolCallResult:
    part, supplier = args.get("part_id"), args.get("supplier","SupplierX")
    days = DATA["LEAD"].get(part,{}).get(supplier)
    return _wrap({"ok":False,"error":"No lead time data","part_id":part,"supplier":supplier}) if days is None else _wrap({"ok":True,"part_id":part,"supplier":supplier,"lead_time_days":days})

for name, fn in [
    ("get_work_order", t_get_work_order),
    ("get_bom", t_get_bom),
    ("check_stock", t_check_stock),
    ("suggest_substitute", t_suggest_substitute),
    ("transfer_options", t_transfer_options),
    ("check_lead_time", t_check_lead_time),
]:
    REG.register(name, fn)

TOOLS_SPECS = [
    {"type":"function","function":{"name":"get_work_order","description":"Fetch work order.","parameters":{"type":"object","properties":{"wo_id":{"type":"string"}},"required":["wo_id"]}}},
    {"type":"function","function":{"name":"get_bom","description":"Get BOM for an asset.","parameters":{"type":"object","properties":{"asset_id":{"type":"string"}},"required":["asset_id"]}}},
    {"type":"function","function":{"name":"check_stock","description":"Check on-hand at site.","parameters":{"type":"object","properties":{"part_id":{"type":"string"},"site":{"type":"string"}},"required":["part_id","site"]}}},
    {"type":"function","function":{"name":"suggest_substitute","description":"Qualified alternates.","parameters":{"type":"object","properties":{"part_id":{"type":"string"}},"required":["part_id"]}}},
    {"type":"function","function":{"name":"transfer_options","description":"Inter-site transfer options.","parameters":{"type":"object","properties":{"part_id":{"type":"string"},"needed_qty":{"type":"integer"},"to_site":{"type":"string"}},"required":["part_id","needed_qty","to_site"]}}},
    {"type":"function","function":{"name":"check_lead_time","description":"Supplier lead time (days).","parameters":{"type":"object","properties":{"part_id":{"type":"string"},"supplier":{"type":"string"}},"required":["part_id"]}}}
]

SYSTEM = (
  "Spare-Parts Planning Agent.\n"
  "Rules:\n"
  "• Use tools for facts; never invent numbers.\n"
  "• If you mention stock, substitutes, transfers, or lead times, you MUST call the relevant tool and include citations with tool call IDs.\n"
  "• No writes; propose an ACTION PLAN only. Ask for approval.\n"
  "• Finish with ONLY: FINAL_JSON: {summary, actions, citations, risks, next_step}.\n"
  "• Do not emit FINAL_JSON until after all needed tool calls."
)

@dataclass
class Budgets:
    max_steps: int = 6
    max_tokens_total: int = 8000
    max_ms: int = 8000

def prune(messages: List[Dict[str, Any]], tail:int=10) -> List[Dict[str, Any]]:
    return messages[:2] + messages[-tail:] if len(messages)>2 else messages

def run_agent(user_text: str, budgets: Budgets = Budgets()) -> str:
    messages: List[Dict[str, Any]] = [
        {"role":"system","content":SYSTEM},
        {"role":"user","content":user_text},
    ]
    total_tokens = 0
    t0 = time.perf_counter()

    for step in range(1, budgets.max_steps+1):
        if (time.perf_counter()-t0)*1000 > budgets.max_ms:
            return "[guardrail] time budget exceeded"

        resp = CLIENT.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            tools=TOOLS_SPECS,
            tool_choice="auto",
            temperature=0.2,
            max_tokens=250,
        )
        usage = getattr(resp, "usage", None)
        if usage:
            total_tokens += (usage.prompt_tokens or 0) + (usage.completion_tokens or 0)
            if total_tokens > budgets.max_tokens_total:
                return "[guardrail] token budget exceeded"

        msg = resp.choices[0].message

        if msg.tool_calls:  # tool phase
            messages.append({"role":"assistant","tool_calls":msg.tool_calls})
            for call in msg.tool_calls:
                name = call.function.name
                args = json.loads(call.function.arguments or "{}")
                result = REG.dispatch(name, args)
                messages.append({
                    "role":"tool",
                    "tool_call_id": call.id,
                    "name": name,
                    "content": result.content
                })
            messages[:] = prune(messages)
            continue

        # text phase
        content = (msg.content or "").strip()
        messages.append({"role":"assistant","content":content})
        if content.startswith("FINAL_JSON:"):
            return content
        messages[:] = prune(messages)

    return "[guardrail] max steps reached without FINAL_JSON"

# ---- Run a live demo query ----
query = "WO-981 needs 2 x PS-200 at Site K by Friday. Propose fastest vs lowest-cost plan."
print(f"[Azure LIVE] scenario={SCENARIO}")
print(run_agent(query))


[Azure LIVE] scenario=TRANSFER


BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[2].role', 'code': None}}